# שלב 1 — הכנת הנתונים (Porto Taxi Trajectories)

מחברת PySpark לעבודה **מקומית על מדגם**. אותו קוד רץ ללא שינוי על DataProc על הדאטה המלא (רק משנים את הנתיבים ו-`USE_SAMPLE=False`).

**מה המחברת עושה:**
1. טעינת ה-CSV ל-Spark
2. פירוק עמודת `POLYLINE` (JSON של `[[lon,lat],...]`) למערך נקודות
3. ניקוי נתונים פגומים — טכני (מחיקה) ומבני (השלמה), שניהם מתועדים
4. חילוץ פיצ'רים גיאומטריים: מספר נקודות, משך נסיעה, מרחק (Haversine)
5. סטטיסטיקות בסיסיות (כולל `approx_percentile`)
6. הכנת פיצ'רים ל-ML: טיפול בחריגים (Winsorize), צמצום skew, קידוד קטגוריאלי, חוסר איזון מחלקות
7. שמירה כ-**Parquet** אחד (פורמט יעיל) — קלט הן לשלב 2 (H3) והן לניתוחי EDA/ML נוספים

> כל החישובים הכבדים נעשים עם פונקציות Spark SQL נייטיביות (higher-order functions על מערכים / Window) — בלי Python UDF — כדי שהכול יתבצע מבוזר ויעיל.


## 1. הקמת SparkSession

In [ ]:
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import ArrayType, DoubleType

spark = (
    SparkSession.builder
    .appName("porto-taxi-stage1")
    .master("local[*]")                 # מקומי. על Dataproc: להסיר את השורה הזו (YARN ינהל)
    .config("spark.sql.shuffle.partitions", "8")   # מקומי קטן; על הקלאסטר העלו ל-200+
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark


## 2. קונפיגורציה (נתיבים, מדגם/מלא, תיבת גבול פורטו, ספי ML)

In [ ]:
USE_SAMPLE = True   # מקומי = True. ריצה מלאה על Dataproc = False

if USE_SAMPLE:
    INPUT_PATH  = "sample_train.csv"
    OUTPUT_PATH = "out/trips_clean_sample.parquet"
else:
    # על Dataproc — מצביעים ל-GCS
    INPUT_PATH  = "gs://YOUR_BUCKET/raw/train.csv"
    OUTPUT_PATH = "gs://YOUR_BUCKET/clean/trips_clean.parquet"

# תיבת גבול גסה של פורטו (lon/lat) — נקודות מחוץ לה הן ככל הנראה שגיאות GPS
LON_MIN, LON_MAX = -8.75, -8.45
LAT_MIN, LAT_MAX = 41.05, 41.25

# סף מהירות לא-הגיונית בין שתי נקודות עוקבות (15 שניות ביניהן)
MAX_KMH = 200.0
MAX_SEG_KM = MAX_KMH * 15 / 3600   # מרחק מקסימלי סביר בקטע אחד (~0.83 ק"מ)

# ספים לשלב הכנת הפיצ'רים ל-ML (סעיפים 8-11 בהמשך)
WINSOR_PCT = 0.95              # חיתוך 5% העליונים של distance/duration/speed
RARE_STAND_THRESHOLD = 0.01    # תחנות שמשמשות פחות מ-1% מהנסיעות מתאחדות ל-"Other_Stand"

print(f"max segment km = {MAX_SEG_KM:.3f}")


## 3. טעינת הדאטה הגולמי

In [ ]:
raw = (
    spark.read
    .option("header", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", False)
    .csv(INPUT_PATH)
)
print("raw rows:", raw.count())
raw.printSchema()
raw.select("TRIP_ID", "TAXI_ID", "MISSING_DATA", "POLYLINE").show(2, truncate=60)


## 4. פירוק POLYLINE למערך נקודות

`POLYLINE` הוא מחרוזת JSON. נמיר אותה למערך `array<array<double>>` עם `from_json` (נייטיבי, מבוזר), כאשר לכל נקודה `p[0]=lon, p[1]=lat`.

In [ ]:
point_schema = ArrayType(ArrayType(DoubleType()))

df = raw.withColumn("points", F.from_json("POLYLINE", point_schema))
df = df.withColumn("n_points", F.size("points"))

df.select("TRIP_ID", "n_points").show(3)


## 5. ניקוי נתונים — טכני ומבני

**5.1 שגיאות טכניות (נזרקות)** — אלו לא אינפורמטיביות ואי אפשר להשלים אותן משמעותית, ולכן (ורק עבור הקטגוריה הזו) זורקים אותן ישירות:

| בעיה | הסיבה שהיא פוגעת בניתוח | הטיפול |
|------|------------------------|--------|
| `MISSING_DATA = True` | הדאטה עצמו מסומן כחסר | זריקה |
| פוליליין ריק / נקודה אחת | אין מסלול לנתח | `n_points >= 2` |
| נקודה מחוץ לתיבת פורטו | שגיאת GPS שתעוות מרחק/מסלול | פילטר min/max בתוך התיבה |

בדיקת "קפיצת" GPS לא-פיזיקלית (`max_segment_km`) מגיעה רק בסעיף 6, אחרי שמחשבים מרחקים.

**5.2 חוסר מבני (מושלם, לא נזרק)** — `ORIGIN_CALL`/`ORIGIN_STAND` חסרים **לפי מבנה**, לא באקראי: `ORIGIN_CALL` מתמלא רק כש-`CALL_TYPE='A'`, ו-`ORIGIN_STAND` רק כש-`CALL_TYPE='B'` (מאומת על הדאטה: null(`ORIGIN_CALL`) מתלכד כמעט לגמרי עם `CALL_TYPE != 'A'` — כלומר MNAR ולא MCAR). זריקת השורות האלה הייתה מוחקת את רוב הדאטה בלי סיבה טובה — במקום זה מקודדים "לא רלוונטי לנסיעה הזו" כקטגוריה מפורשת משלה.

In [ ]:
# 5.1 פילטרים זולים קודם
df = df.filter((F.col("MISSING_DATA") == "False") & (F.col("n_points") >= 2))

# 5.1 (המשך) min/max קואורדינטות לכל נסיעה (לבדיקת תיבת גבול)
df = (
    df
    .withColumn("lon_min", F.expr("array_min(transform(points, p -> p[0]))"))
    .withColumn("lon_max", F.expr("array_max(transform(points, p -> p[0]))"))
    .withColumn("lat_min", F.expr("array_min(transform(points, p -> p[1]))"))
    .withColumn("lat_max", F.expr("array_max(transform(points, p -> p[1]))"))
)
df = df.filter(
    (F.col("lon_min") >= LON_MIN) & (F.col("lon_max") <= LON_MAX) &
    (F.col("lat_min") >= LAT_MIN) & (F.col("lat_max") <= LAT_MAX)
).drop("lon_min", "lon_max", "lat_min", "lat_max")


In [ ]:
# 5.2 מילוי חוסר מבני - קטגוריה מפורשת במקום null
df = df.withColumn(
    "ORIGIN_CALL_filled",
    F.when(F.col("ORIGIN_CALL").isNull(), "0").otherwise(F.col("ORIGIN_CALL"))
)
df = df.withColumn(
    "ORIGIN_STAND_filled",
    F.when(F.col("ORIGIN_STAND").isNull(), "Not_Applicable").otherwise(F.col("ORIGIN_STAND"))
)


## 6. חילוץ פיצ'רים: משך, מרחק, קטע מקסימלי (Haversine נייטיבי)

מרחק Haversine בין נקודות עוקבות מחושב עם `zip_with` על המערך המוסט באחד, ו-`aggregate` לסכימה. הכול ב-Spark SQL — מבוזר, בלי UDF.

נוסחה: `d = 2R·asin(sqrt( sin²(Δφ/2) + cosφ₁·cosφ₂·sin²(Δλ/2) ))`, R=6371 ק"מ.

In [ ]:
R_KM = 6371.0

# ביטוי המחזיר מערך של מרחקי הקטעים העוקבים (ק"מ)
seg_km_expr = f"""
zip_with(
    slice(points, 1, size(points)-1),
    slice(points, 2, size(points)-1),
    (a, b) -> 2 * {R_KM} * asin(sqrt(
        pow(sin(radians(b[1]-a[1])/2), 2) +
        cos(radians(a[1])) * cos(radians(b[1])) * pow(sin(radians(b[0]-a[0])/2), 2)
    ))
)
"""

df = (
    df
    .withColumn("seg_km", F.expr(seg_km_expr))
    .withColumn("distance_km", F.expr("aggregate(seg_km, cast(0.0 as double), (acc, x) -> acc + x)"))
    .withColumn("max_segment_km", F.expr("array_max(seg_km)"))
    .withColumn("duration_sec", (F.col("n_points") - 1) * 15)
    .withColumn("avg_kmh", F.when(F.col("duration_sec") > 0,
                                  F.col("distance_km") / (F.col("duration_sec") / 3600.0)))
).drop("seg_km")

# 6.1 זריקת נסיעות עם קפיצת GPS לא-פיזיקלית
df = df.filter(F.col("max_segment_km") <= MAX_SEG_KM).drop("max_segment_km").cache()

print("clean rows:", df.count())
df.select("TRIP_ID","n_points","duration_sec","distance_km","avg_kmh").show(5)


## 7. סטטיסטיקות בסיסיות

In [ ]:
df.select("n_points", "duration_sec", "distance_km", "avg_kmh").describe().show()

# רבעונים מקורבים (זול ומבוזר) — מתחבר לרעיון ה-Approximate מהפרויקט
df.select(
    F.expr("approx_percentile(distance_km, array(0.5,0.9,0.99))").alias("distance_km_p50_p90_p99"),
    F.expr("approx_percentile(duration_sec, array(0.5,0.9,0.99))").alias("duration_p50_p90_p99"),
    F.countDistinct("TAXI_ID").alias("n_taxis"),
).show(truncate=False)


## 8. טיפול בחריגים (Winsorize)

מסלולים ארוכים/איטיים באמת קיימים ולגיטימיים (בניגוד לשגיאות GPS שכבר סוננו) — אז לא מוחקים אותם, אלא **חותכים** (winsorize) בערך ה-percentile ה-95, כדי שלא יעוותו סטטיסטיקות/מודלים בהמשך. `approxQuantile` הוא action זול, single-pass עם שגיאה חסומה; החיתוך עצמו הוא `when/otherwise` וקטורי לגמרי בלי shuffle.

In [ ]:
def winsorize(df, cols, upper_pct=WINSOR_PCT, relative_error=0.001):
    thresholds = {}
    for c in cols:
        q = df.approxQuantile(c, [upper_pct], relative_error)[0]
        thresholds[c] = q
        df = df.withColumn(f"{c}_wins", F.when(F.col(c) > q, F.lit(q)).otherwise(F.col(c)))
    return df, thresholds

df, winsor_thresholds = winsorize(df, ["distance_km", "duration_sec", "avg_kmh"])
print("winsorizing thresholds (p95):", winsor_thresholds)


## 9. טרנספורמציות לצמצום Skew

`log1p` ו-`sqrt` על העמודות ה-winsorized, למשוך פנימה את הזנב הימני החזק (מרחק/משך נסיעה).

In [ ]:
def add_skew_transforms(df, cols):
    for c in cols:
        src = f"{c}_wins"
        df = df.withColumn(f"{c}_log", F.log1p(F.col(src)))
        df = df.withColumn(f"{c}_sqrt", F.sqrt(F.col(src)))
    return df

df = add_skew_transforms(df, ["distance_km", "duration_sec"])


## 10. קידוד קטגוריאלי

- `CALL_TYPE` — קרדינליות קבועה וזעירה (A/B/C) → one-hot מפורש
- `TAXI_ID` (442 מוניות ייחודיות) — frequency-encoding דרך Window (ספירה מקומית לפרטישן, בלי groupBy+join)
- `ORIGIN_STAND` — תחנות נדירות (<1% מהנסיעות) מתאחדות ל-`Other_Stand` לפני frequency-encoding, כדי לא לייצר רמה כמעט-ייחודית לכל תחנה נדירה

In [ ]:
# 10.1 one-hot ל-CALL_TYPE
for v in ["A", "B", "C"]:
    df = df.withColumn(f"CALL_TYPE_{v}", (F.col("CALL_TYPE") == v).cast("int"))

# 10.2 frequency-encoding ל-TAXI_ID
w_taxi = Window.partitionBy("TAXI_ID")
df = df.withColumn("TAXI_ID_freq", F.count(F.lit(1)).over(w_taxi))

# 10.3 איחוד תחנות נדירות + frequency-encoding
total_rows = df.count()
w_stand = Window.partitionBy("ORIGIN_STAND_filled")
df = df.withColumn("_stand_share", F.count(F.lit(1)).over(w_stand) / F.lit(total_rows))
df = df.withColumn(
    "ORIGIN_STAND_grouped",
    F.when(F.col("_stand_share") < RARE_STAND_THRESHOLD, F.lit("Other_Stand"))
     .otherwise(F.col("ORIGIN_STAND_filled"))
).drop("_stand_share")

w_stand_grouped = Window.partitionBy("ORIGIN_STAND_grouped")
df = df.withColumn("ORIGIN_STAND_freq", F.count(F.lit(1)).over(w_stand_grouped))


## 11. חוסר איזון מחלקות (DAY_TYPE)

ברירת המחדל היא **class weights** (משמר את כל הדאטה — סתם מוריד משקל לתרומת המחלקה הרוב בלוס). `undersample_majority` מוגדרת כאופציה נפרדת, לשימוש רק אם `DAY_TYPE` ישמש בפועל כ-target מפוקח וצריך סט מאוזן פיזית.

In [ ]:
def undersample_majority(df, label_col="DAY_TYPE", seed=42):
    counts = {r[label_col]: r["count"] for r in df.groupBy(label_col).count().collect()}
    if len(counts) < 2:
        return df
    minority_n = min(counts.values())
    fractions = {label: (minority_n / n if n > 0 else 0.0) for label, n in counts.items()}
    return df.sampleBy(label_col, fractions=fractions, seed=seed)

day_type_counts = {r["DAY_TYPE"]: r["count"] for r in df.groupBy("DAY_TYPE").count().collect()}
total, n_classes = sum(day_type_counts.values()), len(day_type_counts)
day_type_weights = {label: total / (n_classes * n) for label, n in day_type_counts.items()}
print("DAY_TYPE counts:", day_type_counts)
print("DAY_TYPE class weights:", day_type_weights)

weight_map = F.create_map(*[F.lit(x) for pair in day_type_weights.items() for x in pair])
df = df.withColumn("class_weight", weight_map.getItem(F.col("DAY_TYPE")))


## 12. בחירת עמודות סופית ושמירה כ-Parquet

הפלט כולל את העמודות הגיאומטריות הבסיסיות (`points`, `n_points`, `duration_sec`, `distance_km`, `avg_kmh`) שעליהן מסתמך שלב 2 (H3), את פיצ'רי ה-ML (winsorize/log/sqrt/encoding/class_weight), וגם את `ORIGIN_CALL`/`ORIGIN_STAND` הגולמיים (לצד הגרסאות המושלמות) — נדרשים לניתוחי EDA שבודקים MNAR על החוסר המקורי.


In [ ]:
trips = df.select(
    "TRIP_ID", "TAXI_ID", "CALL_TYPE", "CALL_TYPE_A", "CALL_TYPE_B", "CALL_TYPE_C",
    "ORIGIN_CALL", "ORIGIN_STAND", "ORIGIN_CALL_filled", "ORIGIN_STAND_grouped",
    "ORIGIN_STAND_freq", "TAXI_ID_freq",
    "TIMESTAMP", "DAY_TYPE", "class_weight",
    "points", "n_points", "duration_sec", "distance_km", "avg_kmh",
    "distance_km_wins", "distance_km_log", "distance_km_sqrt",
    "duration_sec_wins", "duration_sec_log", "duration_sec_sqrt",
    "avg_kmh_wins",
)

(
    trips
    .write
    .mode("overwrite")
    .parquet(OUTPUT_PATH)
)
print("saved to:", OUTPUT_PATH)

# בדיקת קריאה חוזרת
check = spark.read.parquet(OUTPUT_PATH)
print("reloaded rows:", check.count())
check.printSchema()


## הערות להרצה על DataProc (הריצה הסופית)

1. העלו `train.csv` (ועדיף כבר Parquet) ל-GCS bucket.
2. בקובץ קונפיגורציה: `USE_SAMPLE = False` והצביעו ל-`gs://...`.
3. הסירו `.master("local[*]")` — YARN על הקלאסטר ינהל.
4. העלו `spark.sql.shuffle.partitions` ל-200 ומעלה.
5. קלאסטר חסכוני: workers כ-spot/preemptible, `--max-idle`, ולמחוק מיד אחרי הריצה.

**תוצרים של השלב:** DataFrame מתועד (`OUTPUT_PATH`) עם עמודות גיאומטריות + פיצ'רי ML, וסטטיסטיקות בסיסיות.
